In [ ]:
import scanpy as sc
import scvelo as scv
import sys
sys.path.insert(1, "SIRV/")
from main import SIRV
from anndata import AnnData

RNA = sc.read_h5ad("./data/mouse_organogenesis/RNA_adata.h5ad")

embryos = ["SeqFISH_Embryo1_adata.h5ad",
           "SeqFISH_Embryo2_adata.h5ad",
           "SeqFISH_Embryo3_adata.h5ad"]

for f in embryos:
    print(f"\n=== Processing {f} ===")
    
    # Load spatial data
    SeqFISH = sc.read_h5ad(f"./data/mouse_organogenesis/{f}")
    
    # Run SIRV
    SeqFISH_imputed = SIRV(SeqFISH, RNA, n_pv=50, metadata_to_transfer=["celltype"])
    
    # Normalize for scVelo
    scv.pp.normalize_per_cell(SeqFISH_imputed, enforce=True)
    
    # Undo double normalization
    SeqFISH_imputed.X = SeqFISH.to_df()[SeqFISH_imputed.var_names]
    
    # Keep original palette if present
    if hasattr(SeqFISH, "uns"):
        SeqFISH_imputed.uns.update(SeqFISH.uns)
    
    # Compute velocity on imputed layers
    scv.pp.moments(SeqFISH_imputed, n_pcs=50, n_neighbors=30)
    scv.tl.velocity(SeqFISH_imputed, mode="stochastic")
    scv.tl.velocity_graph(SeqFISH_imputed)
    
    # Save
    out = f"./data/mouse_organogenesis/{f.replace('.h5ad','')}_SIRV_velocity.h5ad"
    SeqFISH_imputed.write_h5ad(out)
    print(f"Saved: {out}")

In [ ]:
import scvelo as scv
import scanpy as sc
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
import sys
from anndata import AnnData

# --- Point to SIRV folder ---
sys.path.insert(1, "SIRV/")
from main import SIRV

# --- Load datasets ---
RNA = sc.read_h5ad("./data/mouse_organogenesis/RNA_adata.h5ad")
SeqFISH = sc.read_h5ad("./data/mouse_organogenesis/SeqFISH_AllEmbryos_adata.h5ad")

print("RNA:", RNA)
print("SeqFISH:", SeqFISH)

# --- Run SIRV (imputation + label transfer) ---
# n_pv = 50 is the same as the original SIRV paper
SeqFISH_imputed = SIRV(SeqFISH, RNA, n_pv=50, metadata_to_transfer=["celltype"])

# --- Normalize imputed layers for scVelo ---
scv.pp.normalize_per_cell(SeqFISH_imputed, enforce=True)

# --- Undo double-normalization on X ---
SeqFISH_imputed.X = SeqFISH.to_df()[SeqFISH_imputed.var_names]

# --- Keep original palette (optional) ---
if hasattr(SeqFISH, "uns"):
    SeqFISH_imputed.uns.update(SeqFISH.uns)

# --- Compute RNA velocity using imputed spliced/unspliced layers ---
scv.pp.moments(SeqFISH_imputed, n_pcs=50, n_neighbors=30)
scv.tl.velocity(SeqFISH_imputed, mode="stochastic")
scv.tl.velocity_graph(SeqFISH_imputed)

# --- Spatial velocity plot ---
scv.pl.velocity_embedding_stream(
    SeqFISH_imputed, basis="xy_loc", color="celltype_mapped_refined", size=30
)
plt.gca().invert_yaxis()

# --- UMAP velocity plot ---
scv.pl.velocity_embedding_stream(
    SeqFISH_imputed, basis="umap", color="celltype_mapped_refined", size=30
)

# --- Save final imputed + velocity dataset ---
SeqFISH_imputed.write_h5ad("./data/mouse_organogenesis/SeqFISH_SIRV_full_velocity.h5ad")

print("Saved: SeqFISH_SIRV_full_velocity.h5ad")